In [2]:
%cd ../
%load_ext autoreload
%autoreload 2


/home/christian/bachelor-project


In [3]:
import pandas as pd
import numpy as np


In [4]:
# Load the raw measurements data
df = pd.read_parquet('data/raw_measurements.parquet')
print(f"Original data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


Original data shape: (9620463, 14)
Columns: ['air_pressure', 'air_temperature_2m', 'air_temperature_5cm', 'average_wind_direction', 'average_wind_speed', 'dew_point_temperature', 'id', 'precipitation_duration', 'precipitation_indicator', 'quality_level', 'record_date', 'relative_humidity', 'station_id', 'sum_precipitation_height']


,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,record_date,relative_humidity,station_id,sum_precipitation_height
0,1016.7,15.3,17.7,70.0,2.4,2.3,5938383,0.0,0,3,2025-04-03 10:00:00,41.6,3987,0.0
1,1016.8,15.6,17.7,70.0,2.3,1.7,5938384,0.0,0,3,2025-04-03 10:10:00,38.9,3987,0.0
2,1016.7,15.6,18.1,80.0,2.2,2.2,5938385,0.0,0,3,2025-04-03 10:20:00,40.3,3987,0.0
3,1016.6,16.1,18.9,80.0,3.3,2.1,5938386,0.0,0,3,2025-04-03 10:30:00,39.0,3987,0.0
4,1016.5,15.8,18.6,80.0,4.4,1.4,5938387,0.0,0,3,2025-04-03 10:40:00,37.8,3987,0.0


In [6]:
# Check data types and structure
print("Data info:")
df.info()
print(f"\nNumber of unique stations: {df['station_id'].nunique()}")
print(f"Date range: {df['record_date'].min()} to {df['record_date'].max()}")


Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9620463 entries, 0 to 9620462
Data columns (total 14 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   air_pressure              float64       
 1   air_temperature_2m        float64       
 2   air_temperature_5cm       float64       
 3   average_wind_direction    float64       
 4   average_wind_speed        float64       
 5   dew_point_temperature     float64       
 6   id                        int64         
 7   precipitation_duration    float64       
 8   precipitation_indicator   int64         
 9   quality_level             int64         
 10  record_date               datetime64[ns]
 11  relative_humidity         float64       
 12  station_id                int64         
 13  sum_precipitation_height  float64       
dtypes: datetime64[ns](1), float64(9), int64(4)
memory usage: 1.0 GB

Number of unique stations: 32
Date range: 2020-01-01 00:00:00 to 2025-

In [8]:
# Ensure timestamp is datetime type
df['record_date'] = pd.to_datetime(df['record_date'])

# Sort by station_id and timestamp to ensure proper ordering
df = df.sort_values(['station_id', 'record_date'])

print(f"Data sorted by station_id and record_date")
print(f"First few rows:")
df.head(10)


Data sorted by station_id and record_date
First few rows:


,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,record_date,relative_humidity,station_id,sum_precipitation_height
7657496,1026.6,1.8,-0.1,260.0,2.1,1.0,1,0.0,0,3,2020-01-01 00:00:00,94.2,96,0.0
7657497,1026.6,1.7,-0.3,260.0,1.9,0.9,2,0.0,0,3,2020-01-01 00:10:00,94.4,96,0.0
7657498,1026.7,1.6,-0.5,270.0,2.2,0.8,3,0.0,0,3,2020-01-01 00:20:00,94.1,96,0.0
7657499,1026.8,1.7,0.1,260.0,2.3,1.0,4,0.0,0,3,2020-01-01 00:30:00,95.2,96,0.0
7657500,1026.7,1.7,0.3,260.0,1.9,0.9,5,0.0,0,3,2020-01-01 00:40:00,94.7,96,0.0
7657501,1026.7,1.8,0.7,250.0,1.9,1.1,6,0.0,0,3,2020-01-01 00:50:00,95.2,96,0.0
7657502,1026.8,1.8,0.8,240.0,1.2,1.1,7,0.0,0,3,2020-01-01 01:00:00,94.8,96,0.0
7657503,1026.9,1.9,0.9,250.0,1.5,1.2,8,0.0,0,3,2020-01-01 01:10:00,95.2,96,0.0
7657504,1027.0,1.7,0.7,250.0,1.8,0.9,9,0.0,0,3,2020-01-01 01:20:00,94.3,96,0.0
7657505,1027.1,1.8,1.0,270.0,2.2,1.0,10,0.0,0,3,2020-01-01 01:30:00,94.4,96,0.0


In [10]:
# Create hourly timestamp by flooring to the hour
df['hour'] = df['record_date'].dt.floor('h')

# Get all columns except timestamp and hour for aggregation
value_columns = [col for col in df.columns if col not in ['record_date', 'hour', 'station_id']]

print(f"Columns to aggregate: {value_columns}")


Columns to aggregate: ['air_pressure', 'air_temperature_2m', 'air_temperature_5cm', 'average_wind_direction', 'average_wind_speed', 'dew_point_temperature', 'id', 'precipitation_duration', 'precipitation_indicator', 'quality_level', 'relative_humidity', 'sum_precipitation_height']


In [11]:
# Aggregate to hourly resolution by taking the mean
# Group by station_id and hour, then calculate mean for all value columns
hourly_df = df.groupby(['station_id', 'hour'])[value_columns].mean().reset_index()

# Rename 'hour' back to 'timestamp' for consistency
hourly_df = hourly_df.rename(columns={'hour': 'record_date'})

print(f"Hourly aggregated data shape: {hourly_df.shape}")
print(f"Original data shape: {df.shape}")
print(f"Reduction ratio: {df.shape[0] / hourly_df.shape[0]:.2f}x")
print(f"\nFirst few rows of aggregated data:")
hourly_df.head(10)


Hourly aggregated data shape: (1603820, 14)
Original data shape: (9620463, 15)
Reduction ratio: 6.00x

First few rows of aggregated data:


,station_id,record_date,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,relative_humidity,sum_precipitation_height
0,96,2020-01-01 00:00:00,1026.683333,1.716667,0.033333,260.000000,2.050000,0.950000,3.5,0.0,0.0,3.0,94.633333,0.0
1,96,2020-01-01 01:00:00,1027.000000,1.816667,0.916667,261.666667,1.750000,1.050000,9.5,0.0,0.0,3.0,94.616667,0.0
2,96,2020-01-01 02:00:00,1026.833333,1.650000,1.100000,276.666667,1.716667,1.050000,15.5,0.0,0.0,3.0,95.633333,0.0
3,96,2020-01-01 03:00:00,1026.966667,1.300000,1.050000,271.666667,2.016667,0.950000,21.5,0.0,0.0,3.0,97.483333,0.0
4,96,2020-01-01 04:00:00,1027.116667,1.233333,1.116667,261.666667,2.133333,1.016667,27.5,0.0,0.0,3.0,98.450000,0.0
5,96,2020-01-01 05:00:00,1027.416667,0.850000,0.816667,246.666667,2.283333,0.700000,33.5,0.0,0.0,3.0,98.916667,0.0
6,96,2020-01-01 06:00:00,1027.316667,0.750000,0.750000,258.333333,2.316667,0.666667,39.5,0.0,0.0,3.0,99.400000,0.0
7,96,2020-01-01 07:00:00,1027.483333,0.883333,0.883333,268.333333,1.633333,0.716667,45.5,0.0,0.0,3.0,98.766667,0.0
8,96,2020-01-01 08:00:00,1027.800000,1.150000,1.233333,258.333333,1.300000,0.900000,51.5,0.0,0.0,3.0,98.116667,0.0
9,96,2020-01-01 09:00:00,1027.800000,1.716667,1.966667,241.666667,1.150000,1.100000,57.5,0.0,0.0,3.0,95.600000,0.0


In [12]:
# Verify the aggregation
print("Aggregated data info:")
hourly_df.info()
print(f"\nDate range: {hourly_df['record_date'].min()} to {hourly_df['record_date'].max()}")
print(f"Number of unique stations: {hourly_df['station_id'].nunique()}")


Aggregated data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1603820 entries, 0 to 1603819
Data columns (total 14 columns):
 #   Column                    Non-Null Count    Dtype         
---  ------                    --------------    -----         
 0   station_id                1603820 non-null  int64         
 1   record_date               1603820 non-null  datetime64[ns]
 2   air_pressure              1603820 non-null  float64       
 3   air_temperature_2m        1603820 non-null  float64       
 4   air_temperature_5cm       1603820 non-null  float64       
 5   average_wind_direction    1603820 non-null  float64       
 6   average_wind_speed        1603820 non-null  float64       
 7   dew_point_temperature     1603820 non-null  float64       
 8   id                        1603820 non-null  float64       
 9   precipitation_duration    1603820 non-null  float64       
 10  precipitation_indicator   1603820 non-null  float64       
 11  quality_level             16

In [13]:
# Save the hourly aggregated data
output_path = 'data/hourly_measurements.parquet'
hourly_df.to_parquet(output_path, index=False)
print(f"Hourly aggregated data saved to: {output_path}")


Hourly aggregated data saved to: data/hourly_measurements.parquet


In [14]:
# Verify the saved file
verification_df = pd.read_parquet(output_path)
print(f"Verification - loaded data shape: {verification_df.shape}")
print(f"Columns match: {list(verification_df.columns) == list(hourly_df.columns)}")
print(f"\nSample of loaded data:")
verification_df.head()


Verification - loaded data shape: (1603820, 14)
Columns match: True

Sample of loaded data:


,station_id,record_date,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,relative_humidity,sum_precipitation_height
0,96,2020-01-01 00:00:00,1026.683333,1.716667,0.033333,260.000000,2.050000,0.950000,3.5,0.0,0.0,3.0,94.633333,0.0
1,96,2020-01-01 01:00:00,1027.000000,1.816667,0.916667,261.666667,1.750000,1.050000,9.5,0.0,0.0,3.0,94.616667,0.0
2,96,2020-01-01 02:00:00,1026.833333,1.650000,1.100000,276.666667,1.716667,1.050000,15.5,0.0,0.0,3.0,95.633333,0.0
3,96,2020-01-01 03:00:00,1026.966667,1.300000,1.050000,271.666667,2.016667,0.950000,21.5,0.0,0.0,3.0,97.483333,0.0
4,96,2020-01-01 04:00:00,1027.116667,1.233333,1.116667,261.666667,2.133333,1.016667,27.5,0.0,0.0,3.0,98.450000,0.0
